In [1]:
%load_ext autoreload
%autoreload 2

import logging

import pandas as pd

from checkerduck import log as logsetup
from checkerduck.db.db import init_database
from checkerduck.rules import debug
from checkerduck.model.models import Analysis, AnalysisDomain
from checkerduck.rules.seo_rule import *          # re-run this cell if you ADD a rule class
from checkerduck.db.store import fetched_at_now

# WARNING keeps the table readable; DEBUG to watch individual rules and their SQL.
logsetup.setup(logging.WARNING)

# Let the frontend render the full frame; it scrolls the output area natively,
# which is what makes trackpad scrolling work.
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 250)
pd.set_option("display.width", None)
pd.set_option("display.precision", 2)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## Input: domains from CSV



In [2]:
ROOT = debug.project_root()          # repo root, however the kernel was started

CSV_PATH   = ROOT / "data" / "candidate_domains.csv"
DB_PATH    = ROOT / "checkerduck.db"
NOTES_PATH = ROOT / "notebooks" / "rule_comments.json"
TARGET_ID  = "x"
LIMIT      = 200          # the CSV holds ~785 rows; extraction covered the first 200

init_database(db_path=str(DB_PATH), alembic_ini_path=str(ROOT / "alembic.ini") )

ANALYSIS = Analysis(
        target_id="x",
        name=None,
        status=None,
        created_at=fetched_at_now(),
        completed_at=None,
        processed_domains=None,
        domains=[
            # AnalysisDomain(domain=d, price_usd=0, notes=None) for d in candidate_domains['url'][:200]],
            AnalysisDomain(domain=d, price_usd=0, notes=None) for d in [
               'appsgeyser.io',
               'on-magazine.co.uk',
               'bestutahrealestate.com',
               'englishvers.com',
               'itsupplychain.com',
               'designyourway.net',
               'pc-bottleneck.com',
               'dearblessings.com',
               'newsgram.com',
               'finchannel.com'
            ]],
    )


# Your notes per domain. A plain JSON file, so they survive kernel restarts.
COMMENTS = debug.Comments(NOTES_PATH)

print(f"root     {ROOT}")
print(f"domains  {len(ANALYSIS.domains)} e.g. {ANALYSIS.domains[:3]}")
print(f"notes    {COMMENTS}")


root     /Users/antonpolyakov/dev/checkerduck
domains  10 e.g. [AnalysisDomain(domain='appsgeyser.io', price_usd=0, notes=None), AnalysisDomain(domain='on-magazine.co.uk', price_usd=0, notes=None), AnalysisDomain(domain='bestutahrealestate.com', price_usd=0, notes=None)]
notes    Comments('/Users/antonpolyakov/dev/checkerduck/notebooks/rule_comments.json', 3 notes)


## Coverage: fresh data per domain

Does it have the fresh data for each domain and data type?


In [3]:
domains = [d.domain for d in ANALYSIS.domains]
debug.data_freshness(domains)

,Domain,batch_analysis,ahrefs_org_traffic_country,domain_categories,ahrefs_metrics_history,ahrefs_top_pages,ahrefs_backlinks,anchors_forbidden_words,ahrefs_organic_keywords
0,appsgeyser.io,✅,✅,✅,✅,✅,✅,✅,✅
1,on-magazine.co.uk,✅,✅,✅,✅,✅,✅,✅,✅
2,bestutahrealestate.com,✅,✅,✅,✅,✅,✅,✅,✅
3,englishvers.com,✅,✅,✅,✅,✅,✅,✅,✅
4,itsupplychain.com,✅,✅,✅,✅,✅,✅,✅,✅
5,designyourway.net,✅,✅,✅,✅,✅,✅,✅,❌
6,pc-bottleneck.com,✅,✅,✅,✅,✅,✅,✅,❌
7,dearblessings.com,✅,✅,✅,✅,✅,✅,✅,❌
8,newsgram.com,✅,✅,✅,✅,✅,✅,✅,✅
9,finchannel.com,✅,✅,✅,✅,✅,✅,✅,✅


## Review

Pick a domain, type a note. Saves to `rule_comments.json` on blur or Enter (one file write
per edit, not per keystroke), and re-reads the file on every dropdown change so notes edited
elsewhere show up. The note then appears in the `comment` column of the table below.

The dropdown is in CSV order. After a first run, `debug.review(COMMENTS,
run.table(ascending=True).index)` orders it worst-domain-first instead.


In [4]:

debug.review(COMMENTS, [d.domain for d in ANALYSIS.domains])

## Evaluate

In [5]:
#DOMAINS = ["theglobeandmail.com"]
RULES = [
    DomainRatingRule(),
    OrganicTrafficRule(),
    HistoricalOrganicTrafficRule(),          # weight=0.8, min_avg_traffic=50
    GeographyRule(),
    DomainsInOutLinksRatioRule(),
    SingleTopPageTrafficRule(),
    SpamWordsAnchorsRule(),                  # weight=1.0, spam_threshold=0.1
    ForbiddenWordsAnchorRule(),
    ForbiddenWordsOrganicKeywordsRule(),
    SpamWordsOrganicKeywordsRule(),
    DomainCategoryRule("Technology"),      # embedding rule: workers drops to 1 automatically
]
before = globals().get("run")        # kept so the Diff cell can compare

run = debug.evaluate(ANALYSIS, RULES)

debug.show(run.table(comments=COMMENTS))    # hide_constant=True also drops uninformative rules

12:55:44 WARNING forcing workers=1: ['DomainCategory'] hold a torch model, and concurrent inference aborts the process
domain appsgeyser.io, category ['Technology', 'Technology: Shareware/Freeware', 'Technology']
domain on-magazine.co.uk, category ['Entertainment: Magazines', 'Travel', 'Travel', 'Entertainment']
domain bestutahrealestate.com, category ['Real Estate', 'Real Estate']
domain englishvers.com, category ['Education: Reference', 'Education']
domain itsupplychain.com, category ['Business & Economy', 'Business & Economy: Business']
domain designyourway.net, category ['Internet Communication', 'Internet Communication: Personal Blogs', 'Technology', 'Technology']
domain pc-bottleneck.com, category ['Shopping & Auctions: Shopping', 'Technology', 'Shopping & Auctions', 'Technology']
domain dearblessings.com, category ['Religion: Religion & Spirituality', 'Education: Reference', 'Religion', 'Education']
domain newsgram.com, category ['Entertainment: News & Media', 'Entertainment']
d

,Domain,comment,overall,weighted,critical,errors,DomainRating,OrganicTraffic,HistoricalOrganicTraffic,Geography,DomainsInOutLinksRatio,SingleTopPageTraffic,SpamWordsAnchors,ForbiddenWordsAnchor,ForbiddenWordsOrganicKeywords,SpamWordsOrganicKeywords,DomainCategory
0,designyourway.net,,0.79,0.78,0,2,0.71,0.11,1.0,1.0,0.84,0.82,1.00,0.96,NaN,NaN,0.67
1,dearblessings.com,,0.70,0.70,0,2,0.72,0.16,1.0,1.0,0.59,0.63,0.94,0.92,NaN,NaN,0.39
2,bestutahrealestate.com,,0.66,0.64,2,0,0.71,0.00,1.0,1.0,0.86,0.47,0.94,0.98,0.0,1.0,0.30
3,appsgeyser.io,,0.66,0.66,2,0,0.62,0.92,1.0,0.0,0.33,0.64,0.88,1.00,1.0,0.0,0.83
4,englishvers.com,,0.64,0.64,2,0,0.65,0.00,1.0,1.0,0.62,0.50,0.90,0.94,0.0,1.0,0.47
5,itsupplychain.com,,0.59,0.57,3,0,0.64,0.00,1.0,1.0,1.00,0.54,0.12,0.86,1.0,0.0,0.38
6,finchannel.com,,0.57,0.54,2,0,0.65,0.00,1.0,1.0,1.00,0.40,0.40,0.58,0.9,0.0,0.36
7,pc-bottleneck.com,,0.53,0.54,0,2,0.61,0.13,1.0,1.0,0.21,0.22,0.06,0.94,NaN,NaN,0.63
8,newsgram.com,,0.52,0.49,3,0,0.62,0.00,1.0,0.0,1.00,0.73,0.58,0.40,0.0,1.0,0.39
9,on-magazine.co.uk,,0.40,0.36,5,0,0.51,0.00,1.0,0.0,1.00,0.53,0.06,0.92,0.0,0.0,0.37


## Bulk comment editing

The Review control above covers one domain at a time. For many at once:


In [50]:
# several at once ('' clears a note)
COMMENTS.update({
    "eerone.com": "thin content",
})

# walk the worst domains on stdin; Enter skips, '-' clears, interrupt to stop
# COMMENTS.prompt(run.table(ascending=True).index[:10])

# round-trip through a spreadsheet
# COMMENTS.to_csv('notes.csv');  COMMENTS.read_csv('notes.csv')

COMMENTS

Comments('/Users/antonpolyakov/dev/checkerduck/notebooks/rule_comments.json', 3 notes)

## Drill down

Everything one rule set said about one domain, plus every rule that crashed.

In [6]:
worst = "designyourway.net" #run.table(ascending=True).index[0]      # table() is best-first now
print(worst)
run.for_domain(worst)

designyourway.net


,score,critical,details,error,name,weight,area
Rule,,,,,,,
DomainRating,0.71,False,,,Domain Rating,1.0,authority
OrganicTraffic,0.11,False,,,Organic Traffic,1.0,traffic
HistoricalOrganicTraffic,1.00,False,,,Historical Organic Traffic,0.8,traffic
Geography,1.00,False,Country during rule evalution {country},,Geography,1.0,relevance
DomainsInOutLinksRatio,0.84,False,,,Domains In/Out Links Ratio,0.5,authority
SingleTopPageTraffic,0.82,False,,,Single Top Page Traffic,1.0,relevance
SpamWordsAnchors,1.00,False,,,Spam Words in Anchors,1.0,safety
ForbiddenWordsAnchor,0.96,False,,,Spam Words in Anchors,1.0,safety
ForbiddenWordsOrganicKeywords,NaN,False,,"Traceback (most recent call last):\n File ""/U...",Forbidden Words in Organic Keywords,1.0,safety


In [52]:
fails = run.failures()
print(f"{len(fails)} of {run.scores.size} (domain, rule) evaluations raised")
if not fails.empty:
    print(fails.groupby("rule").size().to_string())
    print("\n" + fails.iloc[0]["error"])

403 of 2200 (domain, rule) evaluations raised
rule
DomainCategory            3
Geography               200
SingleTopPageTraffic    200

Traceback (most recent call last):
  File "/Users/antonpolyakov/dev/checkerduck/checkerduck/rules/debug.py", line 294, in one
    ev = rule.eval(ctx)
  File "/Users/antonpolyakov/dev/checkerduck/checkerduck/rules/seo_rule.py", line 170, in eval
    def get_domain_top_traffic_geographies(target_id: str, domain: str, top_n: int) -> List[str]:
                                                                                       ^^^^
NameError: name 'List' is not defined. Did you mean: 'list'?

